In [1]:
import os
import pickle
import numpy as np
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
import pandas as pd
import random
from typing import Optional
from torch.utils.data import Dataset
from transformers import AutoModel, AutoTokenizer
from tqdm import tqdm
from torch import device
import json
from datasets import load_dataset
from dotmap import DotMap

import sys
sys.path.append('../')
from nrs.data.dataset import NewsRecDataset
from nrs.data.mind import *
from nrs.models.make_model import make_model

/home/users1/godbolai/.conda/envs/thesis/lib/python3.8/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Paths for dev split with 10k users
SRC_10K_USER_PATH = '/mount/arbeitsdaten/tcl/data/mind/MINDlarge_dev/10k_behaviors.csv'
SRC_DEV_NEWS_PATH = '/mount/arbeitsdaten/tcl/data/mind/MINDlarge_dev/news.tsv'
DST_DEV_FINAL_EMBEDDINGS_PATH = '/mount/arbeitsdaten/tcl/tclext/godbolai/codethesis/nrs/candidate_rep/dev_candidate_embeddings.pkl'
CKPT_PATH = '/mount/arbeitsdaten/tcl/tclext/godbolai/codethesis/nrs/experiments/MI_bce_N10/checkpoints/ckpt_0'

os.makedirs(os.path.dirname(DST_DEV_FINAL_EMBEDDINGS_PATH), exist_ok=True)

# Dynamic device selection
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
MODEL = 'uclanlp/newsbert'
SEQ_LEN = 50  

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL)


/home/users1/godbolai/.conda/envs/thesis/lib/python3.8/site-packages/huggingface_hub/file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [3]:
def load_news_data(news_path):
    print("Loading and processing news data...")
    try:
        train_news = pd.read_csv(news_path, sep='\t', header=None)
        print("News data loaded and processed.")
        print("Columns in news data:", train_news.columns)
        return train_news
    except Exception as e:
        print(f"Error loading news data: {e}")
        raise e

def load_user_behaviors(user_path):
    print("Loading and processing user behaviors...")
    try:
        train_user_df = pd.read_csv(user_path)
        print("User behaviors loaded and processed.")
        return train_user_df
    except Exception as e:
        print(f"Error loading user behaviors: {e}")
        raise e

def clean_user_behaviors(train_user_df):
    print("Cleaning user behaviors...")
    train_user_df['history'] = train_user_df['history'].apply(lambda x: x if isinstance(x, str) else "")
    print("User behaviors cleaned.")
    return train_user_df

def extract_unique_candidate_ids(train_user_df):
    print("Extracting unique candidate IDs...")
    candidate_ids = train_user_df['history'].apply(lambda x: x.split() if isinstance(x, str) else []).explode().unique()
    candidate_df = pd.DataFrame({'article_id': candidate_ids})
    unique_candidate_ids = candidate_df['article_id'].unique()
    print(f"Extracted {len(unique_candidate_ids)} unique candidate IDs.")
    return unique_candidate_ids

def tokenize_titles(unique_candidate_titles, tokenizer):
    print("Tokenizing titles...")
    tokenized_titles = {}
    for candidate_id, title in tqdm(unique_candidate_titles.items(), desc="Tokenizing titles"):
        tokenized = tokenizer(title, return_tensors='pt', truncation=True, padding='max_length', max_length=SEQ_LEN)
        tokenized_titles[candidate_id] = tokenized
    return tokenized_titles

'''
def generate_candidate_embeddings(text_encoder, tokenized_titles, device):
    print("Generating candidate embeddings...")
    embeddings = {}
    text_encoder.to(device)  
    text_encoder.eval()  

    for candidate_id, tokens in tqdm(tokenized_titles.items(), desc="Generating candidate embeddings"):
        input_ids = tokens['input_ids'].to(device)
        attention_mask = tokens['attention_mask'].to(device)

        try:
            # Pass input_ids through an embedding layer
            embedding_layer = torch.nn.Embedding(num_embeddings=tokenizer.vocab_size, embedding_dim=768).to(device)
            input_ids_embedded = embedding_layer(input_ids)

            # Expand input_ids_embedded to (B, 1, S, D) and attention_mask to (B, 1, S, 1)
            input_ids_embedded = input_ids_embedded.unsqueeze(1)  # Shape (B, 1, S, 768)
            attention_mask = attention_mask.unsqueeze(1).unsqueeze(-1)  # Shape (B, 1, S, 1)

            inputs = (input_ids_embedded, attention_mask)

            with torch.no_grad():
                outputs = text_encoder(inputs)

            embeddings[candidate_id] = outputs[0].cpu().numpy()  # Extract embeddings from the output tuple
        except Exception as e:
            print(f"Error processing Candidate ID {candidate_id}: {e}")

    return embeddings
'''
def generate_candidate_embeddings(text_encoder, tokenized_titles, device):
    print("Generating candidate embeddings...")
    embeddings = {}
    text_encoder.to(device)  
    text_encoder.eval()  

    for candidate_id, tokens in tqdm(tokenized_titles.items(), desc="Generating candidate embeddings"):
        input_ids = tokens['input_ids'].to(device)
        attention_mask = tokens['attention_mask'].to(device)

        try:
            # Pass input_ids through an embedding layer
            embedding_layer = torch.nn.Embedding(num_embeddings=tokenizer.vocab_size, embedding_dim=768).to(device)
            input_ids_embedded = embedding_layer(input_ids)

            # Expand input_ids_embedded to (B, 1, S, D) and attention_mask to (B, 1, S, 1)
            input_ids_embedded = input_ids_embedded.unsqueeze(1)  # Shape (B, 1, S, 768)
            attention_mask = attention_mask.unsqueeze(1).unsqueeze(-1)  # Shape (B, 1, S, 1)

            inputs = (input_ids_embedded, attention_mask)

            with torch.no_grad():
                outputs = text_encoder(inputs)

            # Adjust the shape to remove unnecessary dimension
            embeddings[candidate_id] = outputs[0].squeeze(1).cpu().numpy()  # Shape (1, 256)
        except Exception as e:
            print(f"Error processing Candidate ID {candidate_id}: {e}")

    return embeddings


def save_embeddings(embeddings, file_path):
    print("Saving embeddings...")
    pd.to_pickle(embeddings, file_path)
    print(f"Embeddings saved to {file_path}.")

In [4]:
# Load the model checkpoint and isolate the text encoder
def load_model_from_ckpt(path: str):
    ckpt = torch.load(path, map_location=torch.device('cpu'))
    cfg = DotMap(ckpt['config'])
    model = make_model(cfg)
    model.load_state_dict(ckpt['state_dict'])
    return model, cfg

def isolate_text_encoder(model):
    text_encoder = model.news_encoder  
    return text_encoder


In [5]:
# Load the model checkpoint and isolate the text encoder
model, model_cfg = load_model_from_ckpt(CKPT_PATH)
text_encoder = isolate_text_encoder(model)

In [6]:
# Step 1: Load and process news data (use dev split)
train_news = load_news_data(SRC_DEV_NEWS_PATH)
print(f"News data shape: {train_news.shape}")
print("Sample news data:")
print(train_news.head())

Loading and processing news data...
News data loaded and processed.
Columns in news data: Index([0, 1, 2, 3, 4, 5, 6, 7], dtype='int64')
News data shape: (72023, 8)
Sample news data:
        0          1                2  \
0  N88753  lifestyle  lifestyleroyals   
1  N23144     health       weightloss   
2  N86255     health          medical   
3  N93187       news        newsworld   
4  N75236     health           voices   

                                                   3  \
0  The Brands Queen Elizabeth, Prince Charles, an...   
1                      50 Worst Habits For Belly Fat   
2  Dispose of unwanted prescription drugs during ...   
3  The Cost of Trump's Aid Freeze in the Trenches...   
4  I Was An NBA Wife. Here's How It Affected My M...   

                                                   4  \
0  Shop the notebooks, jackets, and more that the...   
1  These seemingly harmless habits are holding yo...   
2                                                NaN   
3  Lt. Iv

In [7]:
# Step 2: Load and process user behaviors (use 10k users split)
train_user_df = load_user_behaviors(SRC_10K_USER_PATH)
print(f"User behaviors data shape: {train_user_df.shape}")
print("Sample user behaviors data:")
print(train_user_df.head())

Loading and processing user behaviors...
User behaviors loaded and processed.
User behaviors data shape: (10000, 5)
Sample user behaviors data:
   Unnamed: 0     user                    time  \
0      194100   U40438   11/15/2019 9:08:56 AM   
1      144736  U224154   11/15/2019 7:59:13 AM   
2      253165  U658727  11/15/2019 10:29:08 AM   
3      361012  U332982   11/15/2019 7:52:52 AM   
4       25642  U658041   11/15/2019 3:15:05 PM   

                                             history  \
0                                                NaN   
1  N54360 N119126 N9970 N42718 N29528 N67369 N511...   
2                                N95420 N3581 N46994   
3  N86580 N81970 N95734 N96616 N123480 N50645 N66...   
4  N14678 N125881 N72056 N89539 N34823 N62336 N14...   

                                          impression  
0  N29160-0 N54368-0 N91737-0 N18190-0 N89764-0 N...  
1  N58465-0 N54368-0 N30206-0 N122944-0 N18190-0 ...  
2  N112536-0 N53018-0 N89764-0 N29160-0 N117802-0... 

In [8]:
# Step 3: Clean user behaviors
train_user_df = clean_user_behaviors(train_user_df)
print(f"Cleaned user behaviors data shape: {train_user_df.shape}")


Cleaning user behaviors...
User behaviors cleaned.
Cleaned user behaviors data shape: (10000, 5)


In [9]:
# Step 4: Get unique candidate titles from dev news data
news_id_column = 0 
title_column = 3    

unique_candidate_ids = extract_unique_candidate_ids(train_user_df)
unique_candidate_titles = {idx: title for idx, title in train_news[train_news[news_id_column].isin(unique_candidate_ids)][[news_id_column, title_column]].values}
print(f"Number of unique candidate titles: {len(unique_candidate_titles)}")
print("Sample unique candidate titles:")
for idx, title in list(unique_candidate_titles.items())[:10]:
    print(f"ID: {idx}, Title: {title}")

Extracting unique candidate IDs...
Extracted 22141 unique candidate IDs.
Number of unique candidate titles: 22140
Sample unique candidate titles:
ID: N88753, Title: The Brands Queen Elizabeth, Prince Charles, and Prince Philip Swear By
ID: N93187, Title: The Cost of Trump's Aid Freeze in the Trenches of Ukraine's War
ID: N75236, Title: I Was An NBA Wife. Here's How It Affected My Mental Health.
ID: N99744, Title: How to Get Rid of Skin Tags, According to a Dermatologist
ID: N124534, Title: Should NFL be able to fine players for criticizing officiating?
ID: N59220, Title: It's been Orlando's hottest October ever so far, but cooler temperatures on the way
ID: N40259, Title: Chile: Three die in supermarket fire amid protests
ID: N22273, Title: 50 Foods You Should Never Eat, According to Health Experts
ID: N79856, Title: Instagram Filters with Plastic Surgery-Inspired Effects Could Soon Disappear
ID: N72751, Title: Michigan apple recall: Nearly 2,300 crates could be contaminated with liste

In [10]:
# Step 5: Tokenize the titles
tokenized_titles = tokenize_titles(unique_candidate_titles, tokenizer)
print(f"Number of tokenized titles: {len(tokenized_titles)}")
print("Sample tokenized titles:")
for idx, tokens in list(tokenized_titles.items())[:5]:
    print(f"ID: {idx}, Tokens: {tokens}")

Tokenizing titles...


Tokenizing titles: 100%|██████████| 22140/22140 [00:06<00:00, 3169.75it/s]


Number of tokenized titles: 22140
Sample tokenized titles:
ID: N88753, Tokens: {'input_ids': tensor([[ 101, 1996, 9639, 3035, 3870, 1010, 3159, 2798, 1010, 1998, 3159, 5170,
         8415, 2011,  102,    0,    0,    0,    0,    0,    0,    0,    0,    0,
            0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
            0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
            0,    0]]), 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0]])}
ID: N93187, Tokens: {'input_ids': tensor([[  101,  1996,  3465,  1997,  8398,  1005,  1055,  4681, 13184,  1999,
          1996, 19874,  1997,  5924,  1005,  1055,

In [11]:
# Step 6: Generate candidate embeddings using the text encoder
candidate_embeddings = generate_candidate_embeddings(text_encoder, tokenized_titles, DEVICE)

Generating candidate embeddings...


Generating candidate embeddings: 100%|██████████| 22140/22140 [1:28:44<00:00,  4.16it/s]


In [12]:
# Step 7: Save the final embeddings
save_embeddings(candidate_embeddings, DST_DEV_FINAL_EMBEDDINGS_PATH)

Saving embeddings...
Embeddings saved to /mount/arbeitsdaten/tcl/tclext/godbolai/codethesis/nrs/candidate_rep/dev_candidate_embeddings.pkl.


In [13]:
# Print the number of final representations saved
print(f"Number of final representations saved: {len(candidate_embeddings)}")

Number of final representations saved: 22140
